# 8 — Tool Gateway (Agent Engine + MCP)

**AWS-side equivalent:** AgentCore Gateway — `notebooks/aws/08-agentcore-gateway.ipynb` (MCP gateway, Lambda tool target, invoke through gateway). 
**GCP service:** Vertex AI [Agent Engine](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview) with Python tools shipped inside the deployed `AdkApp`, plus the optional MCP-server-on-Cloud-Run pattern reached from ADK via `MCPToolset`.

There is no first-class "MCP gateway" managed service on GCP today (no equivalent to `AgentCore` gateways). The two functional substitutes:

1. **In-engine tools** — register plain Python callables on an `Agent` and deploy via `AdkApp`/`agent_engines.create`. The Agent Engine runtime hosts the callables and invokes them on each turn.
2. **External MCP server** — stand up an HTTP/SSE MCP server on Cloud Run, then attach it to an ADK agent via `MCPToolset(connection_params=SseServerParams(url=...))`. The MCP server is the gateway-shaped surface.

## Service surface (as of 2026-05-20)

- `from vertexai.agent_engines import AdkApp` — wraps a `google.adk.agents.Agent` for deployment.
- `client.agent_engines.create(agent_engine=adk_app, config={...})` — provisions a `reasoningEngines/*` resource hosting the agent + its in-engine tools.
- `client.agent_engines.async_stream_query(name=..., user_id=..., message=...)` — invoke the deployed agent.
- `from google.adk.tools.mcp_tool import MCPToolset, SseServerParams` — attach an external MCP server to an ADK agent.

Docs: <https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/develop/adk>, <https://google.github.io/adk-docs/tools/mcp/>

## Step 1: Config + client

In [40]:
import subprocess, importlib.util

# cloudpickle is required by agent_engines.create to serialize the AdkApp.
if not importlib.util.find_spec('cloudpickle'):
    print('Installing cloudpickle…')
    subprocess.run(['pip', 'install', 'cloudpickle', '--quiet'], check=True)
    print('done')
else:
    print('cloudpickle already installed')

import json
import os
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

GEMINI_MODEL = gcp.get('model_id', 'gemini-2.5-flash')
STAGING_BUCKET = gcp.get('staging_bucket', f'gs://{PROJECT_ID}-orgiq-staging')
tracker = ResourceTracker()

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = REGION

import vertexai
vertexai.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)
client = vertexai.Client(project=PROJECT_ID, location=REGION)
print(f'project={PROJECT_ID} region={REGION} staging={STAGING_BUCKET}')


cloudpickle already installed
project=hoehncloudconsole region=us-central1 staging=gs://hoehncloudconsole-orgiq-staging


## Step 2: Define tools that the engine will host

Re-using the tool shapes from `06-adk-agent.ipynb`. In-engine tools cannot reach the notebook's network, so we keep the implementations self-contained (here: stubs that the AlloyDB-wired version of these notebooks would replace with `psycopg` reads).

In [ ]:
from google.cloud import bigquery as _bq
import vertexai
from vertexai import rag as _rag

BQ_DATASET = gcp.get('bq_dataset', 'orgiq')
CORPUS_NAME = gcp.get('rag_corpus_name')

_bq_client = _bq.Client(project=PROJECT_ID)


def graph_query(skill: str, limit: int = 5) -> dict:
    """Look up people with a given skill in the org graph.

    Args:
        skill: skill or domain name.
        limit: maximum people to return.
    Returns:
        dict with key 'results': list of {name, title, skill, strength}.
    """
    query = f"""
        SELECT p.name, p.title, s.name AS skill, ps.weight AS strength
        FROM `{PROJECT_ID}.{BQ_DATASET}.people` p
        JOIN `{PROJECT_ID}.{BQ_DATASET}.person_skills` ps ON ps.person_id = p.id
        JOIN `{PROJECT_ID}.{BQ_DATASET}.skills` s ON s.id = ps.skill_id
        WHERE LOWER(s.name) LIKE LOWER(@skill_pattern)
        ORDER BY ps.weight DESC
    """
    job_config = _bq.QueryJobConfig(
        query_parameters=[_bq.ScalarQueryParameter('skill_pattern', 'STRING', f'%{skill}%')]
    )
    rows = list(_bq_client.query(query, job_config=job_config).result())[:limit]
    return {
        'results': [
            {'name': r['name'], 'title': r['title'], 'skill': r['skill'], 'strength': float(r['strength'])}
            for r in rows
        ],
        'query_skill': skill,
    }


def document_search(query: str, num_results: int = 3) -> dict:
    """Search org documents via the RAG corpus (falls back gracefully if corpus absent).

    Args:
        query: natural-language query.
        num_results: top-k.
    """
    if not CORPUS_NAME:
        return {'results': [], 'query': query, 'note': 'no rag_corpus_name in config — run 03-rag-corpus first'}
    try:
        vertexai.init(project=PROJECT_ID, location=REGION)
        resp = _rag.retrieval_query(
            rag_resources=[_rag.RagResource(rag_corpus=CORPUS_NAME)],
            text=query,
            rag_retrieval_config=_rag.RagRetrievalConfig(top_k=num_results),
        )
        contexts = list(resp.contexts.contexts) if hasattr(resp.contexts, 'contexts') else []
        return {
            'results': [
                {
                    'source': getattr(ctx, 'source_uri', ''),
                    'score': float(getattr(ctx, 'distance', 0)),
                    'text': getattr(ctx, 'text', '')[:200],
                }
                for ctx in contexts
            ],
            'query': query,
        }
    except Exception as e:
        return {'results': [], 'query': query, 'error': str(e)}


print('tool registry:', [graph_query.__name__, document_search.__name__])
print(f'  graph_query  → BigQuery {PROJECT_ID}.{BQ_DATASET}')
print(f'  document_search → RAG corpus {CORPUS_NAME or "(not configured)"}')


## Step 3: Wrap them in an ADK Agent + AdkApp

In [42]:
from google.adk.agents import Agent
from google.genai import types
from vertexai.agent_engines import AdkApp

agent = Agent(
    model=GEMINI_MODEL,
    name='orgiq_tools_agent',
    description='Org expertise + document tools, hosted in Agent Engine.',
    instruction=(
        'You are OrgIQ Search. Use graph_query to find people with a skill, '
        'and document_search to find docs. Always cite by name.'
    ),
    tools=[graph_query, document_search],
    generate_content_config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=512),
)

adk_app = AdkApp(agent=agent, enable_tracing=True)
print('✓ AdkApp constructed (tracing on)')

✓ AdkApp constructed (tracing on)


## Step 4: Local smoke test (no deploy)

`AdkApp` works locally before deployment — same instance can be exercised against an in-process runner.

In [43]:
async def local_smoke():
    print('--- local AdkApp stream ---')
    async for event in adk_app.async_stream_query(
        user_id='local-tester',
        message='Who knows CI/CD? Then find related docs.',
    ):
        # event is a dict in local mode.
        kind = list(event.get('content', {}).get('parts', [{}])[0].keys())[0] if event.get('content') else 'event'
        print(f'  [{kind}] {json.dumps(event)[:240]}')

await local_smoke()

--- local AdkApp stream ---
opentelemetry-exporter-otlp-proto-http is not installed. Please call 'pip install google-cloud-aiplatform[agent_engines]'.
proceeding with tracing disabled because not all packages (i.e. `google-cloud-trace`, `opentelemetry-sdk`, `opentelemetry-exporter-gcp-trace`) for tracing have been installed
  [function_call] {"model_version": "gemini-2.5-flash", "content": {"parts": [{"function_call": {"id": "adk-048caf9b-f6ad-4f3f-ba2a-4fa1dee521be", "args": {"skill": "CI/CD"}, "name": "graph_query"}, "thought_signature": "Cv0DAY89a1-X12NwxGufh8KZB27Kzq6OlQdkp
  [function_response] {"content": {"parts": [{"function_response": {"id": "adk-048caf9b-f6ad-4f3f-ba2a-4fa1dee521be", "name": "graph_query", "response": {"results": [{"name": "Dave Park", "title": "Sr Platform Eng", "skill": "CI/CD", "strength": 0.93}], "query_s
  [function_call] {"model_version": "gemini-2.5-flash", "content": {"parts": [{"function_call": {"id": "adk-c48a7f3d-69ef-4278-8c2c-9091532fa0f3", "args

## Step 5: Deploy to Agent Engine

Provisions a `reasoningEngines/*` resource that hosts the agent + its tools. Build typically takes ~3 minutes — comment out and rerun later if you don't want to spend the cycle now.

In [44]:
DEPLOY = True  # flip to False to skip the build

if DEPLOY:
    remote = client.agent_engines.create(
        agent=adk_app,
        config={
            'requirements': [
                'google-cloud-aiplatform[agent_engines,adk]>=1.110.0',
                'google-adk>=1.7.0',
                'cloudpickle',
                'pydantic',
            ],
            'display_name': 'orgiq-tools-engine',
            'description': 'OrgIQ tool-hosting Agent Engine (graph + doc tools).',
            'staging_bucket': STAGING_BUCKET,
        },
    )
    REMOTE_NAME = remote.api_resource.name
    REMOTE_ID = REMOTE_NAME.split('/')[-1]
    gcp['tools_engine_id'] = REMOTE_ID
    Path('config.json').write_text(json.dumps(config, indent=2))
    tracker.track('agent_engine', REMOTE_ID, region=REGION)
    tracker.flush()
    print(f'✓ deployed: {REMOTE_NAME}')
else:
    REMOTE_NAME = None
    print('skipped deploy (DEPLOY=False)')


✓ deployed: projects/289864904502/locations/us-central1/reasoningEngines/3089595238457540608


## Step 6: Invoke the deployed agent

In [47]:
if REMOTE_NAME:
    remote = client.agent_engines.get(name=REMOTE_NAME)
    print('--- remote async_stream_query ---')
    async for event in remote.async_stream_query(
        user_id='notebook-user',
        message='Find people strong in Kubernetes and any related docs.',
    ):
        print(f'  {json.dumps(event)[:240]}')

--- remote async_stream_query ---
  {"model_version": "gemini-2.5-flash", "content": {"parts": [{"function_call": {"id": "adk-478ea201-f635-46ff-923f-95782b8d9e6a", "args": {"skill": "Kubernetes"}, "name": "graph_query"}, "thought_signature": "Ct0CAY89a1-UZuJr4ez6x61ia-QxPSZO
  {"content": {"parts": [{"function_response": {"id": "adk-478ea201-f635-46ff-923f-95782b8d9e6a", "name": "graph_query", "response": {"results": [{"name": "Alice Chen", "title": "Staff Platform Eng", "skill": "Kubernetes", "strength": 0.95}],
  {"model_version": "gemini-2.5-flash", "content": {"parts": [{"text": "Alice Chen is strong in Kubernetes. Related documents include \"GitOps Pipeline Setup\" and \"Service Mesh Adoption\".", "thought_signature": "CtgDAY89a1_S2XPU37rXp5rO2Yl


## Step 7 (optional): Attach an external MCP server via Cloud Run

This is the gateway-shaped pattern: stand up an MCP server (FastMCP or `@modelcontextprotocol/server-stdio` wrapped with HTTP/SSE) on Cloud Run, then point an ADK agent at it.

Deployment sketch (run from a host shell, not the notebook):

```bash
gcloud run deploy orgiq-mcp \
    --source ./mcp_server \
    --region $REGION \
    --no-allow-unauthenticated \
    --set-env-vars=PROJECT_ID=$PROJECT_ID
```

Then in ADK:

In [48]:
MCP_URL = gcp.get('mcp_server_url')  # populate from a Cloud Run deploy first

if MCP_URL:
    from google.adk.tools.mcp_tool import MCPToolset
    from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams

    mcp_toolset = MCPToolset(
        connection_params=SseServerParams(url=MCP_URL),
    )
    mcp_agent = Agent(
        model=GEMINI_MODEL,
        name='orgiq_mcp_agent',
        instruction='Use the MCP-hosted tools to answer org questions.',
        tools=[mcp_toolset],
    )
    print(f'✓ ADK agent wired to MCP server: {MCP_URL}')
else:
    print('Skipping MCP demo — set gcp.mcp_server_url in config.json once a Cloud Run MCP server is deployed.')

Skipping MCP demo — set gcp.mcp_server_url in config.json once a Cloud Run MCP server is deployed.


## Cross-reference vs AWS

| AWS step (`08-agentcore-gateway.ipynb`)       | GCP equivalent here                                                |
|------------------------------------------------|--------------------------------------------------------------------|
| `agentcore.create_gateway(... MCP)`            | (no managed gateway) — replace with **in-engine tools** or **MCP server on Cloud Run** |
| `create_gateway_target(... lambdaTarget)`      | tools shipped inside the `AdkApp` deployed via `agent_engines.create`, OR an MCP server's tool registry |
| `invoke_gateway_tool(...)`                     | `remote.async_stream_query(...)` (Agent Engine drives the tool calls itself) |
| Lambda hosting tool code                       | Cloud Run hosting an MCP server (or in-engine Python in the AdkApp) |
| Strands agent talking to gateway               | ADK agent talking to `MCPToolset(SseServerParams)`                  |

**Uncertainty:** the MCP module path inside `google-adk` has moved a few times — if `from google.adk.tools.mcp_tool import MCPToolset` fails, check `from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset` or `pip show google-adk` for the version's actual layout.